# Tugas 7: Multiple Linear Regression
## Nama: SITI NIKMATUS SHOLIHAH
## NIM: 244107020014

## **Tugas 1: Multiple Linear Regression**

In [12]:
# Import library yang diperlukan
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [25]:
# Load dataset insurance
data = pd.read_csv('../Data Praktikum/insurance.csv')

### 1. Identifikasi variabel-variabel yang akan digunakan sebagai variabel bebas (fitur) dan variabel target (biaya medis personal)

In [14]:
# Eksplorasi data
print("\n=== INFO DATASET ===")
data.info()

print("\n=== DESKRIPSI STATISTIK ===")
print(data.describe())

print("\n=== 5 DATA TERATAS ===")
print(data.head())


=== INFO DATASET ===
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 94.5 KB

=== DESKRIPSI STATISTIK ===
               age          bmi     children       charges
count  1338.000000  1338.000000  1338.000000   1338.000000
mean     39.207025    30.663397     1.094918  13270.422265
std      14.049960     6.098187     1.205493  12110.011237
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.296250     0.000000   4740.287150
50%      39.000000    30.400000     1.000000   9382.033000
75%      51.000000    34.693750

In [15]:
# Identifikasi variabel
print("\n=== IDENTIFIKASI VARIABEL ===")
print("\nVariabel Target (y):")
print("- charges: Biaya medis personal (dalam USD)")

print("\nVariabel Bebas/Fitur (X):")
print("1. age: Umur pemegang polis")
print("2. sex: Jenis kelamin (male/female)")
print("3. bmi: Body Mass Index")
print("4. children: Jumlah anak yang ditanggung")
print("5. smoker: Status perokok (yes/no)")
print("6. region: Wilayah tempat tinggal (northeast, northwest, southeast, southwest)")


=== IDENTIFIKASI VARIABEL ===

Variabel Target (y):
- charges: Biaya medis personal (dalam USD)

Variabel Bebas/Fitur (X):
1. age: Umur pemegang polis
2. sex: Jenis kelamin (male/female)
3. bmi: Body Mass Index
4. children: Jumlah anak yang ditanggung
5. smoker: Status perokok (yes/no)
6. region: Wilayah tempat tinggal (northeast, northwest, southeast, southwest)


### 2. Bagi dataset menjadi data latih (train) dan data uji (test) dengan proporsi yang sesuai

In [16]:
# Encode variabel kategorikal menggunakan One-Hot Encoding
# drop_first=True untuk menghindari multicollinearity (dummy variable trap)
data_encoded = pd.get_dummies(data, columns=['sex', 'smoker', 'region'], drop_first=True)

print("\n=== HASIL ONE-HOT ENCODING ===")
print("Kolom setelah encoding:")
print(data_encoded.columns.tolist())
print(f"\nJumlah kolom sebelum encoding: {data.shape[1]}")
print(f"Jumlah kolom setelah encoding: {data_encoded.shape[1]}")


=== HASIL ONE-HOT ENCODING ===
Kolom setelah encoding:
['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']

Jumlah kolom sebelum encoding: 7
Jumlah kolom setelah encoding: 9


In [17]:
# Pisahkan features (X) dan target (y)
X = data_encoded.drop('charges', axis=1)
y = data_encoded['charges']

print("\n=== PEMISAHAN X DAN Y ===")
print(f"Jumlah features (X): {X.shape[1]}")
print(f"Jumlah samples: {X.shape[0]}")
print(f"Target variable (y): charges")


=== PEMISAHAN X DAN Y ===
Jumlah features (X): 8
Jumlah samples: 1338
Target variable (y): charges


In [18]:
# Split data menjadi training set (80%) dan testing set (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\n=== TRAIN-TEST SPLIT ===")
print(f"Data training: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Data testing: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")


=== TRAIN-TEST SPLIT ===
Data training: 1070 samples (80.0%)
Data testing: 268 samples (20.0%)


### 3. Lakukan feature scaling jika diperlukan

In [ ]:
# Feature Scaling menggunakan StandardScaler
# StandardScaler: mengubah data agar memiliki mean=0 dan standard deviation=1

scaler = StandardScaler()

# Fit dan transform pada training data
X_train_scaled = scaler.fit_transform(X_train)

# Transform (BUKAN fit_transform) pada testing data
# Kita hanya transform menggunakan parameter dari training set
X_test_scaled = scaler.transform(X_test)

print("\n=== FEATURE SCALING ===")
print("StandardScaler telah diterapkan!")
print(f"\nMean dari scaled training data (seharusnya mendekati 0):")
print(np.round(np.mean(X_train_scaled, axis=0), 6))
print(f"\nStd dari scaled training data (seharusnya mendekati 1):")
print(np.round(np.std(X_train_scaled, axis=0), 6))


=== FEATURE SCALING ===
StandardScaler telah diterapkan!

Mean dari scaled training data (seharusnya mendekati 0):
[-0. -0. -0. -0.  0. -0. -0.  0.]

Std dari scaled training data (seharusnya mendekati 1):
[1. 1. 1. 1. 1. 1. 1. 1.]


### 4. Buat model multiple linear regression menggunakan Scikit-Learn

In [20]:
# Buat model Multiple Linear Regression
model = LinearRegression()

print("\n=== PEMBUATAN MODEL ===")
print("Model Multiple Linear Regression berhasil dibuat!")
print(f"Model: {model}")


=== PEMBUATAN MODEL ===
Model Multiple Linear Regression berhasil dibuat!
Model: LinearRegression()


### 5. Latih model pada data latih dan lakukan prediksi pada data uji

In [21]:
# Training model menggunakan data yang sudah di-scale
model.fit(X_train_scaled, y_train)

print("\n=== TRAINING MODEL ===")
print("Model berhasil di-training!")
print(f"\nIntercept (b0): ${model.intercept_:.2f}")
print("\nKoefisien untuk setiap fitur:")

coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.to_string(index=False))


=== TRAINING MODEL ===
Model berhasil di-training!

Intercept (b0): $13346.09

Koefisien untuk setiap fitur:
         Feature  Coefficient
      smoker_yes  9558.481409
             age  3614.975415
             bmi  2036.228123
        children   516.890247
region_southwest  -349.110678
region_southeast  -290.157047
region_northwest  -158.140981
        sex_male    -9.293101


In [22]:
# Prediksi pada data test
y_pred = model.predict(X_test_scaled)

print("\n=== PREDIKSI ===")
print("Prediksi berhasil dilakukan pada data testing!")
print(f"Jumlah prediksi: {len(y_pred)}")


=== PREDIKSI ===
Prediksi berhasil dilakukan pada data testing!
Jumlah prediksi: 268


### 6. Evaluasi model dengan menghitung metrik seperti R-squared, MSE, dan MAE. Tampilkan hasil evaluasi

In [23]:
# Evaluasi model dengan berbagai metrik
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("\n" + "="*60)
print("               HASIL EVALUASI MODEL")
print("="*60)

print(f"\n1. R-squared (R²): {r2:.6f}")
print(f"   → Model dapat menjelaskan {r2*100:.2f}% variasi dalam data")
print(f"   → Semakin mendekati 1, semakin baik model")

print(f"\n2. Mean Squared Error (MSE): {mse:,.2f}")
print(f"   → Rata-rata kuadrat error")
print(f"   → Semakin kecil, semakin baik")

print(f"\n3. Root Mean Squared Error (RMSE): ${rmse:,.2f}")
print(f"   → Akar dari MSE, dalam satuan yang sama dengan target")
print(f"   → Error rata-rata sekitar ±${rmse:,.2f}")

print(f"\n4. Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"   → Rata-rata nilai absolut error")
print(f"   → Prediksi meleset rata-rata ${mae:,.2f}")

print("\n" + "="*60)


               HASIL EVALUASI MODEL

1. R-squared (R²): 0.783593
   → Model dapat menjelaskan 78.36% variasi dalam data
   → Semakin mendekati 1, semakin baik model

2. Mean Squared Error (MSE): 33,596,915.85
   → Rata-rata kuadrat error
   → Semakin kecil, semakin baik

3. Root Mean Squared Error (RMSE): $5,796.28
   → Akar dari MSE, dalam satuan yang sama dengan target
   → Error rata-rata sekitar ±$5,796.28

4. Mean Absolute Error (MAE): $4,181.19
   → Rata-rata nilai absolut error
   → Prediksi meleset rata-rata $4,181.19



In [24]:
# Perbandingan prediksi vs actual untuk 10 sample pertama
comparison = pd.DataFrame({
    'Actual ($)': y_test.values[:10],
    'Predicted ($)': y_pred[:10],
    'Difference ($)': y_test.values[:10] - y_pred[:10],
    'Error (%)': ((y_test.values[:10] - y_pred[:10]) / y_test.values[:10] * 100)
})

print("\n=== PERBANDINGAN ACTUAL VS PREDICTED (10 Sample) ===")
print(comparison.to_string(index=True))


=== PERBANDINGAN ACTUAL VS PREDICTED (10 Sample) ===
    Actual ($)  Predicted ($)  Difference ($)   Error (%)
0   9095.06825    8969.550274      125.517976    1.380066
1   5272.17580    7068.747443    -1796.571643  -34.076475
2  29330.98315   36858.410912    -7527.427762  -25.663742
3   9301.89355    9454.678501     -152.784951   -1.642515
4  33750.29180   26973.173457     6777.118343   20.080177
5   4536.25900   10864.113164    -6327.854164 -139.494993
6   2117.33885     170.280841     1947.058009   91.957790
7  14210.53595   16903.450287    -2692.914337  -18.950125
8   3732.62510    1092.430936     2640.194164   70.732905
9  10264.44210   11218.343184     -953.901084   -9.293258


## Kesimpulan

Model Multiple Linear Regression telah berhasil dibuat dengan langkah-langkah:
1. ✅ Identifikasi variabel bebas (fitur) dan target
2. ✅ Pembagian dataset (80% train, 20% test)
3. ✅ Feature Scaling menggunakan StandardScaler
4. ✅ Pembuatan model LinearRegression
5. ✅ Training dan prediksi
6. ✅ Evaluasi dengan R², MSE, RMSE, dan MAE

Model dapat digunakan untuk memprediksi biaya medis berdasarkan fitur-fitur yang tersedia.